In [1]:
import joblib
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load everything we built in previous weeks
nlp_model = joblib.load('../models/nlp_model.pkl')
vectorizer = joblib.load('../models/vectorizer.pkl')
budget_model = joblib.load('../models/budget_model.pkl')
features = joblib.load('../models/features.pkl')
knowledge_base = joblib.load('../models/knowledge_base.pkl')
index = faiss.read_index('../models/knowledge_index.faiss')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("All models loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All models loaded successfully!


In [2]:
# ---- AGENT 1: PLANNER AGENT ----
# Takes user input and detects event type + creates a plan

def planner_agent(user_input):
    # Detect event type using our Week 3 NLP model
    vec = vectorizer.transform([user_input])
    event_type = nlp_model.predict(vec)[0]
    
    # Create a checklist based on event type
    checklists = {
        'Wedding': [
            '12 months before: Book venue',
            '8 months before: Finalize catering',
            '6 months before: Book photographer',
            '3 months before: Send invitations',
            '1 month before: Confirm all vendors',
            '1 week before: Final headcount',
        ],
        'Corporate': [
            '3 months before: Book venue and AV equipment',
            '2 months before: Finalize agenda and speakers',
            '1 month before: Send invitations',
            '2 weeks before: Confirm catering',
            '1 week before: Technical rehearsal',
        ],
        'Birthday': [
            '1 month before: Book venue',
            '3 weeks before: Send invitations',
            '1 week before: Order cake and decorations',
            '1 day before: Setup venue',
            'Day of: Arrange return gifts',
        ],
        'Anniversary': [
            '2 months before: Book venue',
            '1 month before: Send invitations',
            '1 week before: Arrange flowers and cake',
            '1 day before: Setup decorations',
            'Day of: Coordinate with photographer',
        ],
        'Party': [
            '1 month before: Decide venue and theme',
            '3 weeks before: Send invitations',
            '1 week before: Arrange food and drinks',
            '1 day before: Setup decorations',
            'Day of: Arrange music and entertainment',
        ],
    }
    
    return {
        'event_type': event_type,
        'checklist': checklists[event_type]
    }

print("Planner Agent ready!")

Planner Agent ready!


In [3]:
# ---- AGENT 2: BUDGET AGENT ----
# Takes event details and predicts budget breakdown

def budget_agent(event_type, city, guest_count, duration_days, season, outdoor):
    
    # Encode inputs exactly like we did in Week 2
    event_map = {'Anniversary': 0, 'Birthday': 1, 'Corporate': 2, 'Party': 3, 'Wedding': 4}
    city_map = {'Bangalore': 0, 'Chennai': 1, 'Delhi': 2, 'Hyderabad': 3, 'Mumbai': 4}
    season_map = {'Monsoon': 0, 'Summer': 1, 'Winter': 2}

    input_data = np.array([[
        event_map[event_type],
        city_map[city],
        guest_count,
        duration_days,
        season_map[season],
        1 if outdoor else 0
    ]])

    # Predict total budget
    total_budget = budget_model.predict(input_data)[0]

    # Split budget into categories based on event type
    splits = {
        'Wedding':     {'Venue': 0.30, 'Catering': 0.25, 'Decoration': 0.20, 'Photography': 0.15, 'Miscellaneous': 0.10},
        'Corporate':   {'Venue': 0.40, 'Catering': 0.30, 'AV Equipment': 0.15, 'Miscellaneous': 0.15},
        'Birthday':    {'Venue': 0.35, 'Catering': 0.30, 'Decoration': 0.20, 'Cake': 0.10, 'Entertainment': 0.05},
        'Anniversary': {'Venue': 0.50, 'Decoration': 0.25, 'Gifts': 0.15, 'Photography': 0.10},
        'Party':       {'Venue': 0.35, 'Catering': 0.35, 'Decoration': 0.20, 'Entertainment': 0.10},
    }

    breakdown = {}
    for category, percentage in splits[event_type].items():
        breakdown[category] = int(total_budget * percentage)

    return {
        'total_budget': int(total_budget),
        'breakdown': breakdown
    }

print("Budget Agent ready!")

Budget Agent ready!


In [4]:
# ---- AGENT 3: KNOWLEDGE AGENT ----
# Answers questions using our Week 4 RAG system

def knowledge_agent(query, top_k=2):
    # Convert question to vector
    query_vector = embed_model.encode([query])
    
    # Search FAISS for most relevant entries
    distances, indices = index.search(np.array(query_vector), top_k)
    
    # Return top results
    results = [knowledge_base[i] for i in indices[0]]
    return results

print("Knowledge Agent ready!")

Knowledge Agent ready!


In [5]:
# ---- MASTER ORCHESTRATOR ----
# This is the brain that connects all 3 agents together

def festiva_planner(user_input, city, guest_count, duration_days, season, outdoor):
    print("=" * 60)
    print(f"User request: {user_input}")
    print("=" * 60)

    # Agent 1: Detect event type and get checklist
    plan = planner_agent(user_input)
    event_type = plan['event_type']

    print(f"\nDetected event type: {event_type}")
    print("\nEvent checklist:")
    for item in plan['checklist']:
        print(f"  - {item}")

    # Agent 2: Predict budget
    budget = budget_agent(event_type, city, guest_count, duration_days, season, outdoor)
    print(f"\nPredicted total budget: Rs.{budget['total_budget']:,}")
    print("\nBudget breakdown:")
    for category, amount in budget['breakdown'].items():
        print(f"  - {category}: Rs.{amount:,}")

    # Agent 3: Fetch relevant knowledge
    knowledge = knowledge_agent(f"{event_type} planning tips")
    print("\nPlanning tips:")
    for tip in knowledge:
        print(f"  - {tip}")

    print("=" * 60)

print("Festiva Planner ready!")

Festiva Planner ready!


In [6]:
# The exact scenario from your project brief!
festiva_planner(
    user_input="I want to plan a wedding in Bangalore",
    city="Bangalore",
    guest_count=200,
    duration_days=2,
    season="Winter",
    outdoor=False
)

User request: I want to plan a wedding in Bangalore

Detected event type: Wedding

Event checklist:
  - 12 months before: Book venue
  - 8 months before: Finalize catering
  - 6 months before: Book photographer
  - 3 months before: Send invitations
  - 1 month before: Confirm all vendors
  - 1 week before: Final headcount

Predicted total budget: Rs.1,206,258

Budget breakdown:
  - Venue: Rs.361,877
  - Catering: Rs.301,564
  - Decoration: Rs.241,251
  - Photography: Rs.180,938
  - Miscellaneous: Rs.120,625

Planning tips:
  - Wedding checklist must include: venue, catering, decoration, photographer, videographer, mehendi artist, music and DJ, bridal wear, and invitation cards.
  - Outdoor weddings require backup plans for rain, extra lighting arrangements, and generator backup for power.


C:\Users\shrey\OneDrive\Desktop\festiva-planner\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
